# Cемантический поиск и RAG

## 1. Подгрузка данных

2 модели как бы выбирал llama deepseek, rag (retrival), KNN, system design карточки ВБ цена за еденицу(литры, кг, пары, штуки), sql, attention, self attention,
multi head attention, transformer Q,K,V.   TF-IDF, bagofwords, как оценивал временой ряд типа какая фича играет большую роль, почему не использовал бустинг а именно 
временой ряд. Рэндом форест и бустинг в чём отличие, как идёт разбивание в рэндо форосте, про раг что делать если модель даёт плохие ответы что тюнить будем, какая была изначальная разметка в РАГ

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("/home/andrey/ds_bootcamp/new_Bootcamp_DS/RAG_data/hh_vacancy.csv")

In [3]:
data.head(5)

,id,name,area,salary,salary_range,experience,schedule,employment,description,key_skills,...,визуализация_skills,has_NLP,NLP_skills,has_CV,CV_skills,has_MLOps,MLOps_skills,has_базы_данных,базы_данных_skills,skills_count
0,119496066,Data Scientist,Москва,NaN,NaN,От 1 года до 3 лет,Удаленная работа,Полная занятость,<p><strong>DDX TECH</strong> — технологическое...,"Python, SQL, Алгоритмы и структуры данных, Ста...",...,[],False,[],False,[],False,[],False,[],2
1,119556787,Data Scientist (AI Stylist),Москва,NaN,NaN,От 1 года до 3 лет,Полный день,Полная занятость,"<p>Мы в поиске Data Scientist в команду, заним...","Python, Machine Learning, Deep Learning, NLP",...,[],False,[],True,"['CLIP', 'SigLIP']",True,"['Docker', 'CI/CD', 'Airflow', 'Docker']",False,[],14
2,119382231,Junior ML Engineer / Data scientist (Младший с...,Москва,NaN,NaN,От 1 года до 3 лет,Удаленная работа,Полная занятость,<p><strong>&quot;Инфосистемы Джет&quot;</stron...,"Python, Git, Linux, ML",...,[],False,[],False,[],True,"['MLflow', 'Apache Airflow', 'Docker', 'Git', ...",False,[],11
3,119393658,ML Engineer/Data Scientist,Санкт-Петербург,NaN,NaN,От 1 года до 3 лет,Удаленная работа,Полная занятость,"<p><em>OGPT — технологическая it-компания, ори...","Python, Алгоритмы и структуры данных, Data Min...",...,[],False,[],True,['GAN'],True,"['Docker', 'Docker']",False,[],7
4,119489019,Project/product менеджер ИИ,Москва,NaN,NaN,От 3 до 6 лет,Полный день,Полная занятость,<p><strong>Чем предстоит заниматься:</strong><...,"ML, Project management, data science, Product ...",...,[],False,[],False,[],False,[],False,[],0


In [ ]:
# data.loc[0]

In [ ]:
import json

data['extracted_skills'] = data['extracted_skills'].map(eval)
data['salary'] = data['salary'].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

In [ ]:
!uv pip install -q langchain_qdrant qdrant_client

## 2. Индексация данных в векторной базе данных

![indexing](https://d11qzsb0ksp6iz.cloudfront.net/assets/dff374c348_indexing-in-vector-database.webp)

#### 🎯 **Основные векторные базы данных:**

* 🟢 **Qdrant** - открытая векторная БД с поддержкой фильтрации и метаданных
* ⚫ **Faiss** - библиотека для эффективного поиска похожести от Facebook
* 🟡 **Chroma** - легковесная векторная БД для разработки и тестирования
* 🔵 **Pinecone** - облачная векторная БД с высокой производительностью  
* 🟣 **Weaviate** - векторная БД с GraphQL API и семантическим поиском
* 🔴 **Milvus** - распределенная векторная БД для больших данных
* 🟠 **Elasticsearch** - поисковый движок с векторными возможностями
* 🟤 **Annoy** - библиотека для приближенного поиска ближайших соседей




* В данном примере мы используем **Qdrant** 🟢

![Chroma Logo](https://media.licdn.com/dms/image/v2/D4E12AQGX2VN5jFwZeQ/article-cover_image-shrink_423_752/article-cover_image-shrink_423_752/0/1703693111520?e=1755734400&v=beta&t=1VJux05ttZNXIrzJHy4lL1ifsYoTII42Js-GvC0L384)

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

In [ ]:
client = QdrantClient(
    path='db/qdrant_db',
    )

In [ ]:
client.create_collection(
    collection_name="demo_collection",
    vectors_config=VectorParams(size=768, distance=Distance.COSINE),
)

![langchani](https://daxg39y63pxwu.cloudfront.net/images/blog/langchain/LangChain.webp)

In [ ]:
!uv pip install -q langchain_huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True, 'batch_size':128}

embeddings_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

In [ ]:
len_data = data['clean_description'].map(lambda x: len(x.split(' ')))

In [ ]:
import seaborn as sns

sns.histplot(len_data)

In [ ]:
len_data.describe()

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
data.head(3)

In [ ]:
from uuid import uuid4
from langchain_core.documents import Document

# Создаем документы для Qdrant с UUID
documents = []
id_mapping = {}  # Сохраним соответствие UUID -> original_id

for _, row in data.iterrows():

    content = row['clean_description']


    # Метаданные
    metadata = {
        "employer": row.get('employer', ''),
        "professional_roles": row.get('professional_roles', ''),
        "experience": row.get('experience', ''),
        "area": row.get('area', ''),
        "salary": row.get('salary', ''),
        "schedule": row.get('schedule', ''),
        "extracted_skills": row['extracted_skills'],  # СПИСОК!
        "source": "vacancy",
        "published_at": row.get('published_at', ''),
        "alternate_url": row.get('alternate_url', ''),
        "has_языки": row.get('has_языки', False),
        "has_библиотеки_ML": row.get('has_библиотеки_ML', False),
        "has_обработка_данных": row.get('has_обработка_данных', False),
        "has_визуализация": row.get('has_визуализация', False),
        "has_NLP": row.get('has_NLP', False),
        "has_CV": row.get('has_CV', False),
        "has_MLOps": row.get('has_MLOps', False),
        "has_базы_данных": row.get('has_базы_данных', False),
        "skills_count": row.get('skills_count', 0)
    }

    documents.append(Document(page_content=content, metadata=metadata))


uuids = [str(uuid4()) for _ in range(len(documents))]

print(f"✅ Создано {len(documents)} документов с UUID")

In [ ]:
documents[0]

In [ ]:
from langchain_qdrant import QdrantVectorStore

vector_store = QdrantVectorStore(
    client=client,
    collection_name="demo_collection",
    embedding=embeddings_model
)

In [ ]:
from tqdm import tqdm

batch_size = 256
total_batches = (len(documents) + batch_size - 1) // batch_size


with tqdm(total=total_batches, desc="Добавление батчей в Qdrant") as pbar:
    for i in range(0, len(documents), batch_size):
        batch_docs = documents[i:i+batch_size]
        batch_ids = uuids[i:i+batch_size]

        vector_store.add_documents(documents=batch_docs, ids=batch_ids)
        pbar.update(1)

print(f"✅ {len(documents)} документов добавлено в Qdrant!")

In [ ]:
points, _ = client.scroll(
    collection_name="demo_collection",
    limit=2,
    with_payload=True,
    with_vectors=True
)

In [ ]:
# points[1].payload['metadata']

### 2.1 Сохранение инексированных данных

### ВАЖНО!!!! Если индексировать(получить эмбеддинги тут на коллабе, нужно их выгрузить!!!) Иначе потом можно остаться без них, а это дорогостоящая и не дешевая операция!
### А Так, вы всегда сможете получить эмбеддинги с гугл диска. И сохранить их как локально, так и куда угодно

In [ ]:
# Выгрузка(С коллаба в Drive)
# !cp -r /content/db /content/drive/MyDrive/qdrant_backup/

# # Загрузка(Скопировать базу данных с Drive в Colab)
# !cp -r /content/drive/MyDrive/qdrant_backup /content/backup/



# Далее вы всегда сможете их обратно подгрузить, просто создав нового клиента

# new_client = QdrantClient(path='/content/backup/qdrant_db')

# points, _ = new_client.scroll(
#     collection_name="demo_collection",
#     limit=2,
#     with_payload=True,
#     with_vectors=True  # ← Важно! Включаем векторы
# )

## 3.Семантический поиск

![pure_semantic_search](https://www.tigerdata.com/_next/image?url=https%3A%2F%2Ftimescale.ghost.io%2Fblog%2Fcontent%2Fimages%2F2025%2F01%2FWhat-Is-Semantic-Search-with-Filters-and-How-to-Implement-It-with-pgvector-and-Python_without-filters.png&w=3840&q=100)

In [ ]:
points, _ = client.scroll(
    collection_name="demo_collection",
    limit=2,
    with_payload=True,
    with_vectors=True
)

In [ ]:
points[0].payload['metadata']

In [ ]:
query = "Профессионально играю на ваших нервах"
results_with_scores = vector_store.similarity_search_with_score(
    query,
    k=7
)

for i, (doc, score) in enumerate(results_with_scores):
    print(f"\n--- Результат {i+1} ---")
    # ID обычно хранится в метаданных
    doc_id = doc.metadata.get('_id', 'Нет ID')

    print(f"🆔 ID в базе: {doc_id}")
    print(f"📊 Similarity Score: {score:.4f}")
    print(f"Должность: {doc.metadata.get('professional_roles', 'Не указано')}")
    print(f"Зарплата: {doc.metadata.get('salary', 'Не указано')}")
    print(f"Компания: {doc.metadata.get('employer', 'Не указано')}")
    print(f"Навыки: {doc.metadata.get('extracted_skills', [])}")
    print(f"Вакансия: {doc.metadata.get('alternate_url', [])}")
    print(f"Опыт: {doc.metadata.get('experience', 'Не указано')}")
    print(f"Описание: {doc.page_content[:400]}...")

### 3.1 Семантический поиск с фильтрацией по метаданным

* Cписок и возможности фильтрации - https://qdrant.tech/documentation/concepts/filtering/

![filtered_semantic_search](https://timescale.ghost.io/blog/content/images/size/w1000/2025/01/What-Is-Semantic-Search-with-Filters-and-How-to-Implement-It-with-pgvector-and-Python_with-filters-1.png)

In [ ]:
documents[0].metadata['salary']

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchAny

my_filter = Filter(
    must=[
        FieldCondition(
            key="metadata.extracted_skills",  # Обратите внимание на путь
            match=MatchAny(any=["SQL"])
        ),
        FieldCondition(
            key="metadata.extracted_skills",
            match=MatchAny(any=["Pandas"])
        ),
        FieldCondition(
            key="metadata.extracted_skills",
            match=MatchAny(any=["scikit-learn"])
        ),
        FieldCondition(
            key="metadata.experience",
            match=MatchAny(any=["От 3 до 6 лет"])
        ),
    ],
    # must_not=[
    #     FieldCondition(
    #         key="metadata.salary",
    #         is_null=False
    #         )
    # ]
)

In [ ]:
query = "Фитнесс индустрия"
results_with_scores = vector_store.similarity_search_with_score(
    query,
    filter=my_filter,
    k=5
)

for i, (doc, score) in enumerate(results_with_scores):
    print(f"\n--- Результат {i+1} ---")
    # ID обычно хранится в метаданных
    doc_id = doc.metadata.get('_id', 'Нет ID')

    print(f"🆔 ID в базе: {doc_id}")
    print(f"📊 Similarity Score: {score:.4f}")
    print(f"Должность: {doc.metadata.get('professional_roles', 'Не указано')}")
    print(f"Зарплата: {doc.metadata.get('salary', 'Не указано')}")
    print(f"Компания: {doc.metadata.get('employer', 'Не указано')}")
    print(f"Навыки: {doc.metadata.get('extracted_skills', [])}")
    print(f"Вакансия: {doc.metadata.get('alternate_url', [])}")
    print(f"Опыт: {doc.metadata.get('experience', 'Не указано')}")
    print(f"Описание: {doc.page_content[:400]}...")

### 4.RAG(Retrieval Augmented Generation)

![RAG Pipeline](https://cdn.hashnode.com/res/hashnode/image/upload/v1724944925051/e525c6cb-6a99-4eec-8b47-3dc827ddff25.png)

In [ ]:
# this is an example of a user question (query)
query = 'Играю на баяне'

In [ ]:
results = vector_store.similarity_search_with_score(
    query,
    filter = my_filter,
    k=10
)

In [ ]:
def format_docs(docs):
    """Форматирует документы для передачи в промпт"""
    formatted = []

    for i, doc in enumerate(docs, 1):
        metadata = doc.metadata

        vacancy_info = f"""
        === ВАКАНСИЯ {i} ===
        Должность: {metadata.get('professional_roles', 'Не указано')}
        Компания: {metadata.get('employer', 'Не указано')}
        Опыт: {metadata.get('experience', 'Не указано')}
        Локация: {metadata.get('area', 'Не указано')}
        График: {metadata.get('schedule', 'Не указано')}
        Навыки: {', '.join(metadata.get('extracted_skills', []))}
        Ссылка: {metadata.get('alternate_url', 'Не указано')}

        Описание: {doc.page_content[:300]}...
        """

        formatted.append(vacancy_info)

    return "\n".join(formatted)

print("✅ Функция форматирования создана")

In [ ]:
!uv pip install -q langchain_groq

In [ ]:
import os
import getpass
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter API key for Groq: ")

llm = ChatGroq(
    model="deepseek-r1-distill-llama-70b",
    temperature=0,
    max_tokens=2000
)

In [ ]:
llm.invoke('Как дела? Как тебя зовут?')

In [ ]:
messages = [
    SystemMessage(content="Если тебя спросят как тебя зовут, представляйся Иваном Ивановым и в догонку анекдот про программистов"),
    HumanMessage(content="Привет, как тебя зовут?")
]

answer = llm.invoke(messages).content

print(answer)

In [ ]:
from langchain.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Ты эксперт-аналитик рынка IT-вакансий с многолетним опытом и отличным чувством юмора! 🎯
    Твоя задача - проанализировать предоставленные вакансии и дать профессиональную оценку с долей иронии.

    Стиль анализа:
    - Проводи глубокий анализ, но с легкой иронией над IT-реалиями
    - Используй IT-мемы и шутки там, где это уместно
    - Подмечай забавные особенности вакансий (завышенные требования, смешные формулировки)
    - Давай практические советы с юмором
    - Структурируй ответ с эмодзи и забавными комментариями
    - Отвечай на русском языке живым, но профессиональным тоном

    Помни: юмор должен быть добрым и не оскорбительным. Цель - сделать анализ интересным!

    Если среди вакансий есть что-то особенно забавное - обязательно это отметь! 😄"""),

    ("human", """📊 ДАННЫЕ ДЛЯ АНАЛИЗА (или как говорят в IT - "сырые данные"):
{context}

🎯 ЗАПРОС НА ЭКСПЕРТИЗУ: {question}""")
])

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [ ]:
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# Создаем RAG цепочку
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    } # словарь, в котором ключи - это переменные, которые будут переданы в промпт
    | rag_prompt # промпт для RAG
    | llm # тут можно поставить любую llm-модель
    | StrOutputParser() # для вывода ответа в читаемом виде
)

print("✅ RAG цепочка создана")

In [ ]:
# Тестируем
question = "Умею играть на баяне. Очень люблю музыку и аналитку музыкальную, подскажи, какие вакансии мне подходят, и чего мне не хватает для этих вакансий"
try:
    answer = rag_chain.invoke(question)
    print("🔍 ОТВЕТ RAG:")
    print(answer)
except Exception as e:
    print(f"❌ Ошибка: {e}")

In [ ]:
def demonstrate_rag_step_by_step(query: str):
    """Демонстрация работы RAG цепочки по шагам с промежуточными результатами"""

    print("🔍 ПОШАГОВОЕ ВЫПОЛНЕНИЕ RAG ЦЕПОЧКИ")
    print("=" * 60)
    print(f"📝 Запрос: '{query}'")
    print()

    # ШАГ 1: Получение контекста из векторной БД
    print("📍 ШАГ 1: Поиск похожих документов")
    print("-" * 40)

    # Получаем документы из ретривера
    docs = retriever.invoke(query)
    print(f"🔢 Найдено документов: {len(docs)}")

    for i, doc in enumerate(docs, 1):
        print(f"  📄 Документ {i}:")
        print(f"     Содержимое: {doc.page_content[:100]}...")
        print(f"     Метаданные: {doc.metadata}")
        print()

    # ШАГ 2: Форматирование документов
    print("📍 ШАГ 2: Форматирование контекста")
    print("-" * 40)

    formatted_context = format_docs(docs)
    print(f"📝 Отформатированный контекст:")
    print(f"{formatted_context[:500]}...")
    print()

    # ШАГ 3: Создание входных данных для промпта
    print("📍 ШАГ 3: Подготовка входных данных")
    print("-" * 40)

    input_data = {
        "context": formatted_context,
        "question": query
    }
    print(f"🗂️ Входные данные:")
    print(f"   📋 Контекст: {len(formatted_context)} символов")
    print(f"   ❓ Вопрос: '{input_data['question']}'")
    print()

    # ШАГ 4: Применение промпта
    print("📍 ШАГ 4: Создание финального промпта")
    print("-" * 40)

    formatted_prompt = rag_prompt.format(**input_data)
    print(f"💬 Финальный промпт для LLM:")
    print(f"{formatted_prompt[:800]}...")
    print()

    # ШАГ 5: Вызов LLM
    print("📍 ШАГ 5: Генерация ответа LLM")
    print("-" * 40)

    # Конвертируем промпт в сообщения
    messages = rag_prompt.format_messages(**input_data)
    print(f"📨 Отправляем {len(messages)} сообщений в LLM")

    # Получаем ответ
    response = llm.invoke(messages)
    print(f"🤖 Сырой ответ LLM:")
    print(f"   Тип: {type(response)}")
    print(f"   Содержимое: {response.content[:200]}...")
    print()

    # ШАГ 6: Парсинг ответа
    print("📍 ШАГ 6: Финальная обработка")
    print("-" * 40)

    final_answer = StrOutputParser().parse(response)
    print(f"✅ Финальный ответ:")
    print(f"{final_answer}")

    return final_answer

In [ ]:
demonstrate_rag_step_by_step(query)